# Урок 13 - Память агента с использованием графов знаний Cognee


## Настройка

Этот блокнот демонстрирует, как создать интеллектуального **помощника по программированию** с устойчивой памятью, используя графы знаний [**Cognee**](https://www.cognee.ai/) и **Microsoft Agent Framework** (MAF).

Cognee преобразует неструктурированный текст в структурированный, доступный для запросов граф знаний, основанный на векторных встраиваниях — обеспечивая вашему агенту богатую долгосрочную память с учетом связей.

### Чему вы научитесь
1. **Создавать графы знаний**: Преобразовывать профили разработчиков и лучшие практики в структурированные, доступные для запросов знания.
2. **Интегрировать Cognee с MAF**: Использовать функции `@tool`, чтобы агент MAF мог выполнять запросы в граф знаний Cognee.
3. **Ведение разговоров с сохранением сессии**: Поддерживать контекст в нескольких вопросах в рамках одной сессии.
4. **Долгосрочная память**: Сохранять важные знания между сессиями и извлекать их в новых разговорах.

### Требования
- Python 3.9+
- Redis, запущенный локально (`docker run -d -p 6379:6379 redis`) для управления сессиями
- API-ключ LLM (например, OpenAI) — установить `LLM_API_KEY` в `.env`
- `CACHING=true` в `.env` (требуется для сессий Cognee)
- Проект Azure AI Foundry с развернутой моделью чата
- Переменные `AZURE_AI_PROJECT_ENDPOINT` и `AZURE_AI_MODEL_DEPLOYMENT_NAME` в `.env`
- Авторизация в Azure CLI (`az login`)


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity "cognee[redis]==0.4.0" -q

In [ ]:
import os
from pathlib import Path
from typing import Annotated

from dotenv import load_dotenv

load_dotenv()

os.environ["LLM_API_KEY"] = os.getenv("LLM_API_KEY", "")
os.environ["CACHING"] = os.getenv("CACHING", "true")

import cognee
from cognee.modules.search.types import SearchType

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

print(f"Cognee version: {cognee.__version__}")
print(f"CACHING: {os.environ.get('CACHING')}")

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Типы памяти агента

В этой тетрадке исследуются те же три типа памяти, что и в основной тетрадке Урока 13, но в качестве долговременной памяти используется Cognee:

| Тип памяти | Механизм | Время жизни |
|---|---|---|
| **Рабочая** | `agent.create_session()` (MAF) | Одна беседа |
| **Краткосрочная** | Кэш сессии Cognee (Redis) | Одна сессия |
| **Долгосрочная** | Знаниевая графа Cognee + векторы | Постоянная |

### Архитектура памяти Cognee
```
┌──────────────────────────┐
│      Raw Data            │  (developer profiles, docs, conversations)
└───────────┬──────────────┘
            │  cognee.add() + cognee.cognify()
            ▼
┌──────────────────────────────────────────┐
│  Knowledge Graph + Vector Embeddings     │
└───────────┬──────────────────────────────┘
            │  cognee.search()
            ▼
┌──────────────────┐       ┌────────────────┐
│  MAF Agent       │──────▶│  @tool funcs   │
│  (AgentSession)  │       │  wrapping       │
│                  │       │  cognee.search  │
└──────────────────┘       └────────────────┘
```


## Подготовка хранилища Cognee


In [ ]:
DATA_ROOT = Path('.data_storage').resolve()
SYSTEM_ROOT = Path('.cognee_system').resolve()

DATA_ROOT.mkdir(parents=True, exist_ok=True)
SYSTEM_ROOT.mkdir(parents=True, exist_ok=True)

cognee.config.data_root_directory(str(DATA_ROOT))
cognee.config.system_root_directory(str(SYSTEM_ROOT))

await cognee.prune.prune_data()
await cognee.prune.prune_system(metadata=True)
print("✅ Cognee storage configured and reset")

## Часть 1 — Создание базы знаний

Мы собираем три типа данных для создания комплексной базы знаний для нашего помощника по программированию:

1. **Профиль разработчика** — личная экспертиза и технический бэкграунд
2. **Лучшие практики Python** — Дзен Python с практическими рекомендациями
3. **Исторические беседы** — прошлые сессии вопросов и ответов между разработчиками и AI-помощниками


In [ ]:
developer_intro = (
    "Hi, I'm an AI/Backend engineer. "
    "I build FastAPI services with Pydantic, heavy asyncio/aiohttp pipelines, "
    "and production testing via pytest-asyncio. "
    "I've shipped low-latency APIs on AWS, Azure, and GoogleCloud."
)

python_zen_principles = """
# The Zen of Python: Practical Guide

## Key Principles With Guidance

### 1. Beautiful is better than ugly
Prefer descriptive names, clear structure, and consistent formatting.

### 2. Explicit is better than implicit
Be clear about behavior, imports, and types.

### 3. Simple is better than complex
Choose straightforward solutions first.

### 4. Flat is better than nested
Use early returns to reduce indentation.

## Modern Python Tie-ins
- Type hints reinforce explicitness
- Context managers enforce safe resource handling
- Dataclasses improve readability for data containers
"""

human_agent_conversations = """
"conversations": [
    {
      "topic": "async/await patterns",
      "user_query": "I'm building a web scraper that needs to handle thousands of URLs concurrently. What's the best way to structure this with asyncio?",
      "assistant_response": "Use asyncio with aiohttp, a semaphore to cap concurrency, TCPConnector for connection pooling, and context managers for session lifecycle."
    },
    {
      "topic": "dataclass vs pydantic",
      "user_query": "When should I use dataclasses vs Pydantic models?",
      "assistant_response": "For API input/output, prefer Pydantic: runtime validation, type coercion, JSON serialization. Integrates cleanly with FastAPI."
    },
    {
      "topic": "testing patterns",
      "user_query": "What's the best approach for pytest with async functions?",
      "assistant_response": "Use pytest-asyncio, async fixtures, and an isolated test database or mocks to reliably test async code."
    },
    {
      "topic": "error handling and logging",
      "user_query": "What's the best approach for production-ready error management?",
      "assistant_response": "Centralized error handling with custom exceptions, structured logging, and FastAPI middleware."
    }
  ]
"""

print("✅ Data sources prepared")

In [ ]:
await cognee.add(developer_intro, node_set=["developer_data"])
await cognee.add(human_agent_conversations, node_set=["developer_data"])
await cognee.add(python_zen_principles, node_set=["principles_data"])

await cognee.cognify()
print("✅ Knowledge graph built")

## Визуализация графа знаний

Cognee может отображать интерактивную HTML-визуализацию извлечённых сущностей и связей.


In [ ]:
from cognee import visualize_graph

await visualize_graph('./cognee_graph.html')
print("📊 Graph saved to cognee_graph.html — open it in a browser to explore.")

## Обогащение памяти с помощью Memify

`memify()` анализирует граф знаний и генерирует интеллектуальные правила — выявляя закономерности, лучшие практики и взаимосвязи между концепциями.


In [ ]:
await cognee.memify()
print("✅ Memory enriched with memify")

## Часть 2 — Агент MAF с инструментами Cognee

Теперь мы создадим агента MAF, который может выполнять запросы к графу знаний Cognee с помощью функций `@tool`. Это позволяет агенту использовать всю мощь семантического поиска с учётом графа, сохраняя при этом контекст разговора через сессии.


In [ ]:
@tool(approval_mode="never_require")
async def search_knowledge(
    query: Annotated[str, "Natural-language question to search the knowledge graph"],
) -> str:
    """Search the Cognee knowledge graph for relevant developer knowledge, best practices, and past conversations."""
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
    )
    if not results:
        return "No relevant knowledge found."
    return str(results)


@tool(approval_mode="never_require")
async def search_principles(
    query: Annotated[str, "Question about Python principles or best practices"],
) -> str:
    """Search only the Python principles subset of the knowledge graph."""
    from cognee.modules.engine.models.node_set import NodeSet
    results = await cognee.search(
        query_text=query,
        query_type=SearchType.GRAPH_COMPLETION,
        node_type=NodeSet,
        node_name=["principles_data"],
    )
    if not results:
        return "No relevant principles found."
    return str(results)


print("✅ Cognee tools defined: search_knowledge, search_principles")

In [ ]:
coding_agent = await provider.create_agent(
    name="CodingAssistant",
    instructions=(
        "You are an expert coding assistant with access to a knowledge graph "
        "containing developer profiles, Python best practices, and past conversations.\n\n"
        "WORKFLOW:\n"
        "1. Use search_knowledge() to find relevant information from the full knowledge graph.\n"
        "2. Use search_principles() when the question is specifically about Python best practices.\n"
        "3. Combine retrieved knowledge with your own expertise to give comprehensive answers.\n"
        "4. Reference the developer's known tech stack (FastAPI, asyncio, Pydantic) when relevant."
    ),
)

print("✅ CodingAssistant agent created")

## Рабочая память с сессиями

`AgentSession` (создаваемая через `agent.create_session()`) обеспечивает рабочую память внутри сессии. Агент может ссылаться на предыдущие сообщения, а также запрашивать долгосрочную графовую базу знаний Cognee.


In [ ]:
session = coding_agent.create_session()

response = await coding_agent.run(
    "How does my AsyncWebScraper implementation align with Python's design principles?",
    session=session,
)
print("🤖 Agent:", response)

In [ ]:
response = await coding_agent.run(
    "Based on what you just said, when should I pick dataclasses versus Pydantic for this work?",
    session=session,
)
print("🤖 Agent:", response)
print("\n💡 The agent combined working memory (previous answer) with Cognee's knowledge graph.")

## Новая сессия — долгосрочная память сохраняется

Начало новой сессии очищает рабочую память, но граф знаний Cognee по-прежнему доступен. Агент может извлекать те же долгосрочные знания в совершенно новом разговоре.


In [ ]:
session_2 = coding_agent.create_session()

response = await coding_agent.run(
    "What logging guidance should I follow for incident reviews?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 New session, but the agent still has access to the full Cognee knowledge graph.")

In [ ]:
response = await coding_agent.run(
    "How should variables be named according to Python best practices?",
    session=session_2,
)
print("🤖 Agent:", response)

## Резюме

В этой тетради вы создали помощника по программированию, который объединяет **оперативную память MAF** (`agent.create_session()`) с **долговременным графом знаний Cognee**.

### Чему вы научились
1. **Построение графа знаний**: Cognee обрабатывает неструктурированный текст и создает граф + векторную память.
2. **Обогащение графа с помощью memify**: Выведение фактов и более богатых связей поверх вашего существующего графа.
3. **Интеграция MAF + Cognee**: Функции `@tool` позволяют агенту MAF естественно выполнять запросы к графу Cognee.
4. **Оперативная память + долговременная память**: `AgentSession` (через `agent.create_session()`) обеспечивает контекст сессии, в то время как Cognee предоставляет постоянные знания.
5. **Фильтрованный поиск с NodeSets**: Нацеливание на конкретные подмножества графа знаний (например, только принципы).

### Основные выводы
- **Cognee** преобразует необработанный текст в структурированную память с отношениями — более мощную, чем плоское векторное хранилище.
- **Функции `@tool`** обеспечивают чистое взаимодействие между агентами MAF и внешними системами знаний.
- **`AgentSession`** (через `agent.create_session()`) отделяет контекст конкретного разговора от долговременных знаний.
- Один и тот же граф знаний используется для нескольких сессий и агентов.

### Применение в реальных условиях
- **Помощники для разработчиков**: обзор кода, анализ инцидентов, архитектурные помощники
- **Клиентоориентированные помощники**: агенты поддержки на основе документации продукта, FAQ и заметок CRM
- **Внутренние экспертные помощники**: помощники по политикам, юридическим или вопросам безопасности, работающие с руководствами
- **Объединённые уровни данных**: сочетание структурированных и неструктурированных данных в одном графе для запросов

### Следующие шаги
- Экспериментировать с временной осведомленностью в Cognee
- Определить OWL онтологию для качества графа в доменной области
- Добавить циклы обратной связи от пользователей для улучшения поиска с течением времени
- Масштабировать до мультиагентных систем, использующих общий слой памяти Cognee


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от ответственности**:  
Этот документ был переведен с использованием сервиса автоматического перевода [Co-op Translator](https://github.com/Azure/co-op-translator). Несмотря на наши усилия обеспечить точность, пожалуйста, учитывайте, что автоматические переводы могут содержать ошибки или неточности. Оригинальный документ на его родном языке следует считать авторитетным источником. Для критически важной информации рекомендуется профессиональный человеческий перевод. Мы не несем ответственности за любые недоразумения или неправильные толкования, возникшие в результате использования данного перевода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
